# 07 · Detector YOLO: entrenamiento a la vista en Jupyter

Mismo *fine-tuning* que `scripts/detector_entrenar.py` (YOLO26n preentrenado en COCO → nuestras bebidas), pero con
`model.train(...)` **dentro del notebook**: se ve cada época con sus pérdidas (`box_loss`, `cls_loss`, `dfl_loss`)
y las métricas de validación (P, R, mAP50, mAP50-95), TensorBoard en vivo, las curvas, la matriz de confusión, el
error de **conteo** y las cajas dibujadas.

**Importante: esta PC no tiene GPU.** El entrenamiento completo en CPU tarda entre 1,5 y 6 días por variante
(medido en `docs/detector.md`). Por eso hay dos modos:

| `MODO` | qué hace | tiempo aprox. en esta PC |
|---|---|---|
| `'demo'` | 256 fotos de train (64 de val/test), 3 épocas | ~10 min: sirve para **ver el proceso**, el mAP sale bajo |
| `'completo'` | todas las fotos, 60-80 épocas (`HIPER`) | días en CPU → usar `04_detector_colab.ipynb` en Colab |

Kernel: `inventariovision`. Las corridas quedan en `runs/detector/<variante>_jupyter_<modo>/` y **no** pisan los
pesos del sistema (salvo que se ponga `INSTALAR = True` en modo completo).

In [ ]:
# 0. Configuración: lo único que hace falta tocar
VARIANTE = 'multiclase'   # 'multiclase' (8 clases: marca + envase) | '1clase' (todo es 'producto')
MODO = 'demo'             # 'demo' | 'completo'
CON_SINTETICOS = False    # sumar al train el lote de estantes sintéticos más reciente (datos/sinteticos/lote_*)
N_DEMO, EPOCAS_DEMO = 256, 3
HILOS = 4                 # hilos de torch en CPU
INSTALAR = False          # modo completo: copiar best.pt a modelos/detector/detector_<variante>.pt (lo usa el sistema)
FORZAR_CPU = False        # modo completo sin GPU: hay que confirmarlo a propósito (son días)

In [ ]:
# 1. Entorno
import os, sys, json, time, shutil
from pathlib import Path
print('Python:', sys.executable)
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO))
os.environ['TENSORBOARD_BINARY'] = str(Path(sys.executable).parent / 'Scripts' / 'tensorboard.exe')

import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image
from IPython.display import display, Markdown, Image as IImage
import torch
from ultralytics import YOLO, settings
from ultralytics.utils import torch_utils

import scripts.detector_entrenar as de
import scripts.detector_evaluar as dv
from src.config import DIR_DATOS, DIR_MODELOS, UMBRAL_DET
pd.set_option('display.width', 200)

DEVICE = '0' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu' and HILOS:
    torch_utils.NUM_THREADS = HILOS; torch.set_num_threads(HILOS)
if MODO == 'completo' and DEVICE == 'cpu' and not FORZAR_CPU:
    raise SystemExit('MODO completo sin GPU: son días de CPU. Usar MODO="demo", entrenar en Colab '
                     '(notebooks/04_detector_colab.ipynb) o poner FORZAR_CPU=True a propósito.')
NOMBRE = f'{VARIANTE}_jupyter_{MODO}'
NOMBRES = de.nombres_variante(VARIANTE)
print('torch', torch.__version__, '· device:', DEVICE, '· corrida:', de.DIR_RUNS / NOMBRE)

## 1. Los datos

Dataset fusionado por el agente Datos (`datos/fusion/<variante>/`), en formato YOLO: por cada imagen un `.txt`
con una línea por objeto `clase xc yc w h` (centro, ancho y alto **normalizados 0-1**). El prefijo del archivo
(`alias__...`) dice de qué dataset público viene.

In [ ]:
d = de.dir_dataset(VARIANTE)
filas = [{'split': s, 'origen': p.name.split('__')[0]} for s in ('train', 'val', 'test') for p in de.imagenes_de(d / 'images' / s)]
comp = pd.DataFrame(filas).value_counts().unstack('split', fill_value=0)[['train', 'val', 'test']]
comp.loc['TOTAL'] = comp.sum()
display(comp)

# Objetos por clase en train
cuenta = pd.Series(0, index=NOMBRES)
for lbl in (d / 'labels' / 'train').glob('*.txt'):
    for k in dv.cajas_reales(lbl):
        cuenta.iloc[k if VARIANTE == 'multiclase' else 0] += 1
cuenta.plot.barh(title='Objetos etiquetados en train', figsize=(6, 3)); plt.show()

In [ ]:
# Cuatro imágenes de train con sus cajas reales
def dibujar(ax, ruta_img, ruta_lbl, color='lime', titulo=''):
    im = Image.open(ruta_img); W, H = im.size
    ax.imshow(im); ax.axis('off')
    lineas = ruta_lbl.read_text().splitlines() if ruta_lbl.exists() else []
    for l in lineas:
        k, xc, yc, w, h = map(float, l.split()[:5])
        ax.add_patch(patches.Rectangle(((xc - w / 2) * W, (yc - h / 2) * H), w * W, h * H, fill=False, color=color, lw=1.5))
        if VARIANTE == 'multiclase':
            ax.text((xc - w / 2) * W, (yc - h / 2) * H, NOMBRES[int(k)], fontsize=6, color='black', backgroundcolor=color)
    ax.set_title(titulo or f'{ruta_img.name[:28]}… · {len(lineas)} objetos', fontsize=8)

rng = np.random.default_rng(1)
ej = rng.choice(de.imagenes_de(d / 'images' / 'train'), 4, replace=False)
fig, ejes = plt.subplots(1, 4, figsize=(18, 5))
for a, p in zip(ejes, ej):
    dibujar(a, p, d / 'labels' / 'train' / (p.stem + '.txt'))
plt.tight_layout(); plt.show()

### `data.yaml`

YOLO lee la lista de imágenes de un `data.yaml`. Se arma uno propio en `runs/detector/tmp/<corrida>/` (no se toca
`datos/`). En modo `demo` se **copia** un subconjunto estratificado por dataset de origen.

In [ ]:
sint = de.buscar_sinteticos(VARIANTE) if CON_SINTETICOS else []
if CON_SINTETICOS:
    print('Sintéticos:', sint or 'NO encontrados (se entrena sin ellos)')
DATA_YAML, RESUMEN = de.armar_data_yaml(VARIANTE, de.DIR_RUNS / 'tmp' / NOMBRE,
                                        n_prueba=N_DEMO if MODO == 'demo' else 0, sinteticos=sint,
                                        max_por_origen=4000 if (VARIANTE == '1clase' and MODO == 'completo') else None)
print('Imágenes usadas:', RESUMEN)
print(DATA_YAML.read_text(encoding='utf-8')[:800])

## 2. Hiperparámetros y aumento de datos

- `epochs`/`patience`: corta si `mAP50-95` de val no mejora en `patience` épocas y se queda con la mejor (`best.pt`).
- `imgsz=640`: la imagen se reescala a 640 px de lado (con relleno, sin deformar).
- **Mosaic**: junta 4 fotos en una → más objetos chicos y contextos por lote. Se apaga en las últimas 10 épocas.
- `flipud=0`: nunca hay botellas cabeza abajo. `fliplr` solo en 1clase (en multiclase el logo espejado no existe).
- `hsv_*`: variaciones de color/brillo moderadas.

In [ ]:
PARAMS = dict(de.HIPER, **de.AUMENTO[VARIANTE])
PARAMS.update(data=str(DATA_YAML), device=DEVICE, workers=2, single_cls=(VARIANTE == '1clase'),
              project=str(de.DIR_RUNS), name=NOMBRE, exist_ok=True)
if MODO == 'demo':
    PARAMS.update(epochs=EPOCAS_DEMO, close_mosaic=1)
elif VARIANTE == 'multiclase':
    PARAMS.update(epochs=80)          # igual que en Colab
pd.Series(PARAMS, name='valor').to_frame()

## 3. TensorBoard (en vivo)

Correr antes de entrenar. Se actualiza al cerrar cada época (pestaña *SCALARS*: `train/box_loss`, `metrics/mAP50(B)`, …).
Si no se ve dentro del notebook: http://localhost:6007.

In [ ]:
settings.update({'tensorboard': True})
%load_ext tensorboard
%tensorboard --logdir "{de.DIR_RUNS}" --port 6007

## 4. Entrenamiento

Por cada época Ultralytics muestra una barra con las tres pérdidas de entrenamiento:
- `box_loss`: qué tan bien ajustan las cajas (posición y tamaño).
- `cls_loss`: qué tan bien acierta la clase.
- `dfl_loss`: precisión fina de los bordes de la caja.

y, al terminar la época, la validación: **P** (de lo marcado, cuánto era real), **R** (de lo real, cuánto
encontró), **mAP50** (precisión media con solapamiento ≥ 50 %) y **mAP50-95** (promedio exigiendo de 50 % a 95 %).

In [ ]:
ruta_base = DIR_MODELOS / 'yolo26n.pt'
modelo = YOLO(str(ruta_base))
cron = de.Cronometro(HILOS if DEVICE == 'cpu' else None)
cron.registrar(modelo)
t0 = time.perf_counter()
modelo.train(**PARAMS)
SEG = time.perf_counter() - t0
SAVE_DIR = Path(modelo.trainer.save_dir)
BEST = SAVE_DIR / 'weights' / 'best.pt'
print(f'\n{SEG / 60:.1f} min · corrida en {SAVE_DIR}')
display(pd.DataFrame(cron.epocas))

## 5. Curvas de entrenamiento

In [ ]:
display(IImage(str(dv.graficar_curvas(SAVE_DIR / 'results.csv')), width=1000))
pd.read_csv(SAVE_DIR / 'results.csv').rename(columns=str.strip).round(4)

In [ ]:
# Gráficos que guarda Ultralytics: lotes de entrenamiento (con mosaic) y validación real vs. predicho
for nombre in ('train_batch0.jpg', 'val_batch0_labels.jpg', 'val_batch0_pred.jpg'):
    if (SAVE_DIR / nombre).exists():
        display(Markdown(f'**{nombre}**')); display(IImage(str(SAVE_DIR / nombre), width=800))

## 6. Evaluación en val y test (con `best.pt`)

En multiclase, la matriz de confusión tiene una fila/columna extra **background**: una caja real que no se
detectó (columna de la clase, fila background) o una detección que no corresponde a nada real.

In [ ]:
best = YOLO(str(BEST))
METRICAS = {}
for split in ('val', 'test'):
    m = best.val(data=str(DATA_YAML), split=split, imgsz=PARAMS['imgsz'], batch=PARAMS['batch'], device=DEVICE,
                 workers=2, plots=True, project=str(SAVE_DIR), name=f'eval_{split}', exist_ok=True, verbose=False)
    METRICAS[split] = {'mAP50': round(float(m.box.map50), 4), 'mAP50_95': round(float(m.box.map), 4),
                       'P': round(float(m.box.mp), 4), 'R': round(float(m.box.mr), 4)}
    if VARIANTE == 'multiclase' and split == 'test':
        display(pd.Series({m.names[int(c)]: round(float(v), 3) for c, v in zip(m.box.ap_class_index, m.box.maps[m.box.ap_class_index])},
                          name='mAP50-95 por clase (test)').to_frame().T)
display(pd.DataFrame(METRICAS).T)
(SAVE_DIR / 'metricas_jupyter.json').write_text(json.dumps(METRICAS, indent=2), encoding='utf-8')
cmn = SAVE_DIR / 'eval_test' / 'confusion_matrix_normalized.png'
if cmn.exists():
    display(IImage(str(cmn), width=700))

## 7. Lo que le importa al sistema: el **conteo**

Para cada foto de test: objetos reales vs. cajas detectadas con el umbral del sistema (`UMBRAL_DET`).
- **MAE**: error absoluto medio por foto (cuántos productos de más o de menos).
- **exacto_%**: fotos donde el conteo fue exacto. **sesgo** > 0 = cuenta de más.
- Línea base (YOLO de COCO sin entrenar, clase *bottle*): MAE 9.54 · 12.2 % exactas.

In [ ]:
from src.vision.detector import Detector
dir_test = (de.DIR_RUNS / 'tmp' / NOMBRE) if MODO == 'demo' else d
imgs_test = de.imagenes_de(dir_test / 'images' / 'test')[:300]
detector = Detector(BEST, conf=UMBRAL_DET, imgsz=PARAMS['imgsz'], hilos=HILOS if DEVICE == 'cpu' else None)
filas = []
for p in imgs_test:
    reales = dv.cajas_reales(dir_test / 'labels' / 'test' / (p.stem + '.txt'))
    cajas = detector.detectar(p)
    fila = {'imagen': p.name, 'origen': p.name.split('__')[0], 'n_real': len(reales), 'n_det': len(cajas)}
    if VARIANTE == 'multiclase':
        pred = [c.clase for c in cajas]
        fila['error_por_clase_abs'] = sum(abs(pred.count(n) - reales.count(k)) for k, n in enumerate(NOMBRES))
    filas.append(fila)
PI = pd.DataFrame(filas); PI['error'] = PI.n_det - PI.n_real
tabla = pd.DataFrame([{'grupo': 'TOTAL', **dv.resumir_conteo(PI.error, PI.n_real)}]
                     + [{'grupo': o, **dv.resumir_conteo(g.error, g.n_real)} for o, g in PI.groupby('origen')])
display(tabla)
PI.plot.scatter('n_real', 'n_det', alpha=.4, figsize=(5, 5), title='Conteo por foto (test)')
lim = max(PI.n_real.max(), PI.n_det.max()) + 1
plt.plot([0, lim], [0, lim], 'k--', lw=1); plt.show()

## 8. Cajas reales (verde) vs. detectadas (rojo) en fotos de test

In [ ]:
ej = [dir_test / 'images' / 'test' / n for n in PI.sort_values('n_real', ascending=False).imagen.head(4)]
fig, ejes = plt.subplots(1, len(ej), figsize=(18, 5))
for a, p in zip(np.atleast_1d(ejes), ej):
    lbl = dir_test / 'labels' / 'test' / (p.stem + '.txt')
    cajas = detector.detectar(p)
    dibujar(a, p, lbl, titulo=f'real {len(dv.cajas_reales(lbl))} · detectado {len(cajas)}')
    for c in cajas:
        a.add_patch(patches.Rectangle((c.x, c.y), c.w, c.h, fill=False, color='red', lw=1.2))
plt.tight_layout(); plt.show()

In [ ]:
# 9. Instalar los pesos para el sistema (solo modo completo y con INSTALAR = True)
if MODO == 'completo' and INSTALAR:
    de.DIR_PESOS.mkdir(parents=True, exist_ok=True)
    destino = de.DIR_PESOS / f'detector_{VARIANTE}.pt'
    shutil.copy2(BEST, destino)
    print('Pesos instalados en', destino)
else:
    print('No se instala nada. Pesos de esta corrida:', BEST)